In [15]:
# ==========================================
# SEL 1: LOAD BERKAS MASTER RAW
# ==========================================
import pandas as pd
import os

# Membaca data mentah hasil scraping/API
raw_data_path = "../data/raw/all_locations.csv"
if os.path.exists(raw_data_path):
    df = pd.read_csv(raw_data_path)
    print(f"Berhasil memuat data. Ukuran data asli: {df.shape}")
else:
    print(f"Waduh, file tidak ditemukan di {raw_data_path}. Pastikan path-nya benar ya!")

df.head()

Berhasil memuat data. Ukuran data asli: (4368, 8)


,time,location,wave_height,wind_speed_10m,precipitation,visibility,ocean_current_velocity,sea_surface_temperature
0,2026-05-20T00:00,Pacitan,1.60,3.5,0.0,11460.0,0.5,28.0
1,2026-05-20T01:00,Pacitan,1.60,3.0,0.0,14120.0,0.5,28.0
2,2026-05-20T02:00,Pacitan,1.60,4.0,0.0,14200.0,0.5,28.0
3,2026-05-20T03:00,Pacitan,1.58,4.1,0.0,14200.0,0.5,28.0
4,2026-05-20T04:00,Pacitan,1.58,2.9,0.0,11640.0,0.5,28.0


In [16]:
# ==========================================
# SEL 2: DATA PREPROCESSING (TRANSFORMASI WAKTU & IMPUTASI AMAN)
# ==========================================
# 1. Konversi kolom waktu ke tipe datetime
df["time"] = pd.to_datetime(df["time"])

# 2. Wajib SORTING duluan berdasarkan lokasi dan waktu sebelum imputasi!
df = df.sort_values(by=["location", "time"]).reset_index(drop=True)

# 3. Imputasi Forward Fill secara aman per-lokasi agar tidak bocor antar kota
features_to_impute = [
    "wave_height", "wind_speed_10m", "precipitation", 
    "visibility", "ocean_current_velocity", "sea_surface_temperature"
]

for col in features_to_impute:
    # Menggunakan ffill() per kelompok lokasi
    df[col] = df.groupby("location")[col].ffill()
    
print("Preprocessing dan sorting selesai secara aman!")

Preprocessing dan sorting selesai secara aman!


In [17]:
# ==========================================
# SEL 3: FEATURE ENGINEERING (KONSTRUKSI FITUR LAG RUN-TIME)
# ==========================================
# Memberikan 'ingatan' 1-3 jam ke belakang untuk model regresi
features_to_lag = [
    "wave_height", "wind_speed_10m", "precipitation", 
    "visibility", "ocean_current_velocity", "sea_surface_temperature"
]

for col in features_to_lag:
    df[f"{col}_lag1"] = df.groupby("location")[col].shift(1)
    df[f"{col}_lag2"] = df.groupby("location")[col].shift(2)
    df[f"{col}_lag3"] = df.groupby("location")[col].shift(3)

# Hilangkan rekaman bernilai kosong di awal garis waktu tiap lokasi akibat geser jendela lag
df_processed = df.dropna().reset_index(drop=True)

print(f"Feature Engineering selesai. Ukuran data setelah dropna: {df_processed.shape}")

Feature Engineering selesai. Ukuran data setelah dropna: (4125, 26)


In [23]:
# ==========================================
# SEL 4: SIMPAN HASIL PEMROSESAN AKHIR
# ==========================================
# Tentukan path folder tujuan
folder_path = r"d:\lentera-laut\src\data\processed"

# Membuat folder secara otomatis jika belum ada di direktori
os.makedirs(folder_path, exist_ok=True)

# FIXED: Simpan 'df_processed' (bukan 'df' yang masih mengandung NaN)
output_file_path = os.path.join(folder_path, "modeling_ready.csv")
df_processed.to_csv(output_file_path, index=False)

print("Sempurna! File siap pakai berhasil disimpan di:", output_file_path)

Sempurna! File siap pakai berhasil disimpan di: d:\lentera-laut\src\data\processed\modeling_ready.csv


In [22]:
# Menghitung total nilai kosong di setiap kolom
missing_summary = df_processed.isnull().sum()

# Menampilkan kolom yang HANYA memiliki missing value > 0
print("Jumlah Missing Value per Kolom:")
print(missing_summary[missing_summary > 0])

Jumlah Missing Value per Kolom:
Series([], dtype: int64)
